In [1]:
import os
import kagglehub

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import pandas as pd

# Download latest version
path = kagglehub.dataset_download("nicolacarrassi/ava-aesthetic-visual-assessment")

print("Path to dataset files:", path)

csv_path = os.path.join(path, "ground_truth_dataset.csv")
image_dir = os.path.join(path, "images")

C:\Users\User\anaconda3\envs\py311-deeplearning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\nicolacarrassi\ava-aesthetic-visual-assessment\versions\1


In [2]:
df = pd.read_csv(csv_path)

print(df.head(), flush=True)
print(df.columns, flush=True)

   image_num  vote_1    vote_2    vote_3    vote_4    vote_5    vote_6  \
0     953417     0.0  0.000000  0.000000  0.040323  0.258065  0.403226   
1     953777     0.0  0.023438  0.015625  0.023438  0.101562  0.312500   
2     953756     0.0  0.015625  0.023438  0.070312  0.273438  0.390625   
3     954195     0.0  0.008197  0.057377  0.213115  0.459016  0.188525   
4     953903     0.0  0.008065  0.032258  0.040323  0.266129  0.403226   

     vote_7    vote_8    vote_9   vote_10  
0  0.185484  0.080645  0.024194  0.008065  
1  0.273438  0.164062  0.062500  0.023438  
2  0.156250  0.039062  0.015625  0.015625  
3  0.049180  0.008197  0.000000  0.016393  
4  0.137097  0.072581  0.024194  0.016129  
Index(['image_num', 'vote_1', 'vote_2', 'vote_3', 'vote_4', 'vote_5', 'vote_6',
       'vote_7', 'vote_8', 'vote_9', 'vote_10'],
      dtype='str')


In [3]:
import os
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class AVADataset(Dataset):
    VOTE_COLS = [
        'vote_1', 'vote_2', 'vote_3', 'vote_4', 'vote_5',
        'vote_6', 'vote_7', 'vote_8', 'vote_9', 'vote_10'
    ]

    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_id = row['image_num']

        label = torch.tensor(
            row[self.VOTE_COLS].values.astype('float32')
        )
        label = label / label.sum().clamp_min(1e-6)

        image_path = os.path.join(
            self.image_dir,
            f"{int(image_id)}.jpg"
        )

        try:
            if not os.path.exists(image_path):
                raise FileNotFoundError(f"Missing: {image_path}")
        
            image = Image.open(image_path).convert("RGB")
            
            if self.transform:
                image = self.transform(image)
                
        except (OSError, FileNotFoundError, Exception) as e:
            print(f"Skipping corrupted image {image_id}: {e}")
            return self.__getitem__((idx + 1) % len(self.df))

        return image, label

In [4]:
import torch.nn as nn
from torchvision.models import vgg16, VGG16_Weights

weights = VGG16_Weights.DEFAULT

model = vgg16(weights=weights)

for p in model.features.parameters():
    p.requires_grad = False

for p in model.features[24:].parameters():
    p.requires_grad = True

model.classifier[6] = nn.Linear(4096, 10)

In [5]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df), flush=True)

204406 25551 25551


In [7]:
train_dataset = AVADataset(
    train_df,
    image_dir,
    transform=train_transform
)

val_dataset = AVADataset(
    val_df,
    image_dir,
    transform=eval_transform
)

test_dataset = AVADataset(
    test_df,
    image_dir,
    transform=eval_transform
)

batch_size = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

In [8]:
images, labels = next(iter(train_loader))

print(images.shape, flush=True)
print(labels.shape, flush=True)
print(labels[0].sum(), flush=True)

torch.Size([32, 3, 224, 224])
torch.Size([32, 10])
tensor(1.)


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EMDLoss(nn.Module):
    def __init__(self, r=2):
        super(EMDLoss, self).__init__()
        self.r = r

    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1)

        cdf_pred = torch.cumsum(pred, dim=1)
        cdf_target = torch.cumsum(target, dim=1)

        samplewise_emd = torch.mean(
            torch.abs(cdf_pred - cdf_target) ** self.r,
            dim=1
        )

        return torch.mean(samplewise_emd)

In [10]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model = model.to(device)

print_batch = 100

# Hyper-parameters
num_epochs = 100  # students should train 1 epoch because they will use cpu
learning_rate = 1e-4
learning_rate_after_unfreeze = 1e-5

# Loss and optimizer
criterion = EMDLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3#, verbose=True
)

unfreeze_epoch = 3

def evaluate(data_loader):
    model.eval()
    val_loss_sum = 0.0
    val_mae_sum = 0.0
    num_samples = 0

    scores = torch.arange(1, 11, device=device, dtype=torch.float32)
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            
            cur_batch_size = images.size(0)
            num_samples += cur_batch_size

            # EMD loss
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * cur_batch_size

            # Mean score MAE
            pred_dist = torch.softmax(outputs, dim=1)
            pred_mean = (pred_dist * scores).sum(dim=1)
            true_mean = (labels * scores).sum(dim=1)
            mae = torch.abs(pred_mean - true_mean).sum()
            val_mae_sum += mae.item()

    return val_loss_sum / num_samples, val_mae_sum / num_samples

# Train the model
total_step = len(train_loader)
current_lr = learning_rate

best_val_mae = float('inf')

print("Starting training...", flush=True)
for epoch in range(num_epochs):

    if epoch == unfreeze_epoch:
        print("Unfreezing VGG16 Block 5 (layers 24+)", flush=True)
        for p in model.features[24:].parameters():
            p.requires_grad = True
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate_after_unfreeze)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    model.train()
    train_loss_sum = 0.0
    num_samples = 0

    for batch_index, (images, labels) in enumerate(train_loader):
        # print(images.shape, flush=True)
        images = images.to(device)  # "images" = "inputs"
        labels = labels.to(device)  # "labels" = "targets"

        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        loss.backward()
        optimizer.step()

        cur_batch_size = images.size(0)
        train_loss_sum += loss.item() * cur_batch_size
        num_samples += cur_batch_size

        if (batch_index + 1) % print_batch == 0:
            train_loss = train_loss_sum / num_samples
            print("Epoch [{}/{}], Step [{}/{}] Loss: {:.4f}"
                  .format(epoch + 1, num_epochs, batch_index + 1, total_step, train_loss), flush=True)

    val_loss, val_mae = evaluate(val_loader)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(model.state_dict(), './vgg16_best.ckpt')
    
    print(
        f'Epoch [{epoch+1}/{num_epochs}] '
        f'Validation Loss: {val_loss:.4f} '
        f'Validation MAE: {val_mae:.4f}'
    , flush=True)

    scheduler.step(val_mae)

# Save the model checkpoint
#torch.save(model.state_dict(), './vgg16_final.ckpt')

model.load_state_dict(torch.load('./vgg16_best.ckpt'))
_, test_mae = evaluate(test_loader)
print('MAE value of the model on the test images: {}'.format(test_mae), flush=True)

Starting training...
Epoch [1/100], Step [100/6388] Loss: 0.0115
Epoch [1/100], Step [200/6388] Loss: 0.0107


C:\Users\User\anaconda3\envs\py311-deeplearning\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch [1/100], Step [300/6388] Loss: 0.0104
Epoch [1/100], Step [400/6388] Loss: 0.0103
Epoch [1/100], Step [500/6388] Loss: 0.0100
Epoch [1/100], Step [600/6388] Loss: 0.0099
Epoch [1/100], Step [700/6388] Loss: 0.0098
Epoch [1/100], Step [800/6388] Loss: 0.0097
Epoch [1/100], Step [900/6388] Loss: 0.0096
Epoch [1/100], Step [1000/6388] Loss: 0.0095
Epoch [1/100], Step [1100/6388] Loss: 0.0094
Epoch [1/100], Step [1200/6388] Loss: 0.0094
Epoch [1/100], Step [1300/6388] Loss: 0.0093
Epoch [1/100], Step [1400/6388] Loss: 0.0093
Epoch [1/100], Step [1500/6388] Loss: 0.0092
Epoch [1/100], Step [1600/6388] Loss: 0.0092
Epoch [1/100], Step [1700/6388] Loss: 0.0092
Epoch [1/100], Step [1800/6388] Loss: 0.0091
Epoch [1/100], Step [1900/6388] Loss: 0.0091
Epoch [1/100], Step [2000/6388] Loss: 0.0091
Epoch [1/100], Step [2100/6388] Loss: 0.0091
Epoch [1/100], Step [2200/6388] Loss: 0.0090
Epoch [1/100], Step [2300/6388] Loss: 0.0090
Epoch [1/100], Step [2400/6388] Loss: 0.0090
Epoch [1/100], St

KeyboardInterrupt: 

In [16]:
from scipy.stats import pearsonr, spearmanr

def calculate_all_metrics(y_true_dist, y_pred_dist):
    scores = torch.arange(1, 11).float().to(y_pred_dist.device)
    
    true_means = torch.sum(y_true_dist * scores, dim=1)
    pred_means = torch.sum(y_pred_dist * scores, dim=1)
    
    true_std = torch.sqrt(torch.sum(y_true_dist * (scores**2), dim=1) - true_means**2)
    pred_std = torch.sqrt(torch.sum(y_pred_dist * (scores**2), dim=1) - pred_means**2)

    lcc_mean, _ = pearsonr(pred_means.cpu().numpy(), true_means.cpu().numpy())
    srcc_mean, _ = spearmanr(pred_means.cpu().numpy(), true_means.cpu().numpy())

    lcc_std, _ = pearsonr(pred_std.cpu().numpy(), true_std.cpu().numpy())
    srcc_std, _ = spearmanr(pred_std.cpu().numpy(), true_std.cpu().numpy())

    true_labels = (true_means >= 5.0).float()
    pred_labels = (pred_means >= 5.0).float()
    accuracy = (true_labels == pred_labels).float().mean().item()

    cdf_true = torch.cumsum(y_true_dist, dim=1)
    cdf_pred = torch.cumsum(y_pred_dist, dim=1)
    emd = torch.norm(cdf_true - cdf_pred, p=1, dim=1).mean().item() / 10

    return {
        "Accuracy": accuracy * 100,
        "LCC_mean": lcc_mean,
        "SRCC_mean": srcc_mean,
        "LCC_std": lcc_std,
        "SRCC_std": srcc_std,
        "EMD": emd
    }


In [19]:
import torch
import torch.nn.functional as F
import numpy as np

model.load_state_dict(torch.load('./vgg16_best.ckpt'))
all_metrics = []

with torch.no_grad():
    for images, target_dist in test_loader:
        images = images.to(device)
        target_dist = target_dist.to(device)
        
        logits = model(images)
        
        pred_dist = F.softmax(logits, dim=1)
        
        batch_metrics = calculate_all_metrics(target_dist, pred_dist)
        all_metrics.append(batch_metrics)

final_results = {}
for key in all_metrics[0].keys():
    final_results[key] = np.mean([m[key] for m in all_metrics])

print("--- 최종 테스트 결과 ---")
for key, value in final_results.items():
    print(f"{key}: {value:.4f}")

--- 최종 테스트 결과 ---
Accuracy: 77.3060
LCC_mean: 0.6125
SRCC_mean: 0.5873
LCC_std: 0.2123
SRCC_std: 0.2065
EMD: 0.0498
